<a href="https://colab.research.google.com/github/prithwis/GeoSim/blob/main/gsWorldDataStatic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

In [1]:
!pip -q install pymupdf
!wget -O gsLLM.py -q https://raw.githubusercontent.com/prithwis/GeoSim/refs/heads/main/gsUtils/gsLLM.py
!wget -O gsWorldFacts.py -q https://raw.githubusercontent.com/prithwis/GeoSim/refs/heads/main/gsUtils/gsWorldFacts.py                  # all

from gsLLM import *
from gsWorldFacts import *

from pathlib import Path
import time

In [2]:
import importlib
import gsLLM, gsWorldFacts
importlib.reload(gsLLM)
importlib.reload(gsWorldFacts)

<module 'gsWorldFacts' from '/content/gsWorldFacts.py'>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

WORLDFACTS = "/content/drive/MyDrive/Prithwis/GeoSim/Silicon/WorldFacts"

!ls -l "{WORLDFACTS}"

Mounted at /content/drive
total 16
drwx------ 2 root root 4096 Nov 19 07:16 Markdown
drwx------ 2 root root 4096 Nov 21 10:39 PDF_Original_1
drwx------ 2 root root 4096 Nov 21 10:39 PDF_Original_2
-rw------- 1 root root 3421 Nov 19 07:34 Silicon_Stack_Source_Manifest.md


## Static World Data
URLs stored as PDF files that are converted to markdown files

In [4]:
# ------------------------------------------------
# Use OpenAI
# ------------------------------------------------
provider = "openai"

#model = "gpt-5.6-luna"
model = "gpt-6-luna"
#model = "gpt-6.1-sol"
#model = "gpt-6-astra"

# ------------------------------------------------
# Use Claude
# ------------------------------------------------
#!pip install -q anthropic

#provider = "claude"
#model = "claude-haiku-5-5"
#model = "claude-sonnet-5-5"

# ------------------------------------------------
# Authenticate wit LLM provider using API keys stored as Colab Secret
# ------------------------------------------------
client = authenticateLLM(provider=provider)

OPENAI | User: Calcutta | Email: prithwis@yantrajaal.com | ID: user-sNAljKNmBf2igtAV2kUuqhoF


In [11]:
#PDF_DIR = f"{WORLDFACTS}/PDF_Original_1"
PDF_DIR = f"{WORLDFACTS}/PDF_Original_2"
MD_DIR  = f"{WORLDFACTS}/Markdown"

!ls -l "{PDF_DIR}"

total 9690
-rw------- 1 root root 1708713 Nov 19 07:22 03-BCG_Strengthening-the-Global-Semiconductor-Supply-Chain.pdf
-rw------- 1 root root 8213160 Nov 19 07:23 05-EnergyandAI.pdf


In [6]:
from IPython.display import display, Markdown
from pathlib import Path

manifest = Path(WORLDFACTS) / "Silicon_Stack_Source_Manifest.md"
#display(Markdown(manifest.read_text(encoding="utf-8")))

In [12]:
ROLE_DIR = Path(WORLDFACTS).parent / "Roles"
Role_PDFReader = ((ROLE_DIR / "RoleFile_PDFCleaner.txt").read_text(encoding="utf-8").strip())
#print(Role_PDFReader)

In [13]:

from pathlib import Path

pdf_files = sorted(Path(PDF_DIR).glob("*.pdf"))
md_dir = Path(MD_DIR)
md_dir.mkdir(parents=True, exist_ok=True)

print(f"Found {len(pdf_files)} PDF files\n")

md_files = []
failed_files = []

for i, source_PDF in enumerate(pdf_files, 1):

    print("=" * 70)
    print(f"[{i}/{len(pdf_files)}] Processing: {source_PDF.name}")

    try:
        md_path = GetWorldFile(
            client,
            Role_PDFReader,
            source_PDF,
            MD_DIR,
            model,
            provider
        )

        md_path = Path(md_path)

        if not md_path.exists():
            raise FileNotFoundError(
                f"Markdown output not created: {md_path}"
            )

        if md_path.stat().st_size == 0:
            raise ValueError("Generated Markdown is empty")

        md_files.append(md_path)

        print(f"SUCCESS: {md_path.name}")
        print(f"Markdown size: {md_path.stat().st_size:,} bytes")

    except Exception as e:
        failed_files.append((source_PDF.name, str(e)))
        print(f"FAILED: {source_PDF.name}")
        print(f"Reason: {e}")

print("\n" + "=" * 70)
print(f"Completed: {len(md_files)} of {len(pdf_files)} files")
print(f"Failed: {len(failed_files)}")

for name, reason in failed_files:
    print(f"  {name}: {reason}")


Found 2 PDF files

[1/2] Processing: 03-BCG_Strengthening-the-Global-Semiconductor-Supply-Chain.pdf
Reading: 03-BCG_Strengthening-the-Global-Semiconductor-Supply-Chain.pdf
Pages   : 9
Batches : 1
Cleaning 1/1 (9,238 characters)
OPENAI | gpt-6-luna | Effort: medium | In: 2,633 | Cached: 0 | Out: 3,144 | Reasoning: 837 | Time: 19.1s
Output: 9,906 characters
Created  : 03BCGStrengtheningth_261121.md
Markdown : 10,063 characters
SUCCESS: 03BCGStrengtheningth_261121.md
Markdown size: 10,128 bytes
[2/2] Processing: 05-EnergyandAI.pdf
Reading: 05-EnergyandAI.pdf
Pages   : 304
Batches : 6
Cleaning 1/6 (149,210 characters)
OPENAI | gpt-6-luna | Effort: medium | In: 32,275 | Cached: 0 | Out: 28,081 | Reasoning: 271 | Time: 129.7s
Output: 138,791 characters
Cleaning 2/6 (148,527 characters)
OPENAI | gpt-6-luna | Effort: medium | In: 32,832 | Cached: 0 | Out: 27,930 | Reasoning: 336 | Time: 115.8s
Output: 140,201 characters
Cleaning 3/6 (148,656 characters)
OPENAI | gpt-6-luna | Effort: medium | I

In [14]:
!ls -l "{MD_DIR}"

total 1055
-rw------- 1 root root 141288 Nov 21 10:47 01MappingTheSemiCond_261121.md
-rw------- 1 root root  83073 Nov 21 10:48 02ChipLandscape_261121.md
-rw------- 1 root root  10128 Nov 21 10:52 03BCGStrengtheningth_261121.md
-rw------- 1 root root  87257 Nov 21 10:49 04ShivakumarExportCo_261121.md
-rw------- 1 root root 756796 Nov 21 11:03 05EnergyandAI_261121.md


In [ ]:
#mdFile = Path(MD_DIR) / "Copy05EnergyandAI_261121.md"
#display(Markdown(mdFile.read_text(encoding="utf-8")))

In [15]:
for f in md_files:
    print(f.name)

03BCGStrengtheningth_261121.md
05EnergyandAI_261121.md
